# SemEval-2026 Task 13 Subtask A: Linear + NB-SVM + Handcrafted Tree Ensemble v6

Trong notebook này, tôi giữ lại hai nhánh sparse tuyến tính từ phiên bản trước và bổ sung thêm một nhánh tree-based trên bộ handcrafted features giàu thông tin hơn. Mục tiêu của tôi là khai thác đồng thời token-level signal từ TF-IDF và style-level signal từ cấu trúc mã nguồn để tăng độ đa dạng khi ensemble.

Lý do tôi chọn cấu hình này:
- nhánh sparse linear ở phiên bản trước đã khá mạnh, nên dư địa cải thiện chủ yếu đến từ một nguồn tín hiệu khác
- tree model trên handcrafted features có bias khác với TF-IDF, phù hợp để blend
- ở bước chọn cấu hình cuối, tôi vẫn ưu tiên hiệu quả trên `test_sample`, nhưng dùng thêm validation như một tín hiệu ổn định để tránh bám quá sát vào các blend mong manh


In [1]:
import os
import gc
import re
import glob
import time
import warnings
from collections import Counter

import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')

from joblib import Parallel, delayed
from scipy.sparse import csr_matrix, hstack, vstack
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import f1_score, classification_report
from sklearn.preprocessing import StandardScaler

HAVE_LGBM = False
HAVE_HGB = False

try:
    import lightgbm as lgb
    from lightgbm import LGBMClassifier
    HAVE_LGBM = True
except Exception as e:
    print('LightGBM not available, will fall back to HistGradientBoostingClassifier:', e)

if not HAVE_LGBM:
    try:
        from sklearn.ensemble import HistGradientBoostingClassifier
        HAVE_HGB = True
    except Exception as e:
        raise RuntimeError('Neither LightGBM nor HistGradientBoostingClassifier is available') from e

print('Imports OK')
print('HAVE_LGBM =', HAVE_LGBM)


Imports OK
HAVE_LGBM = True


## 1. Nạp dữ liệu

Ở bước này tôi đọc toàn bộ các split cần thiết (`train`, `validation`, `test`, `test_sample`) và kiểm tra nhanh kích thước cũng như phân bố nhãn của từng tập. Việc nhìn sớm vào prior của từng split giúp tôi thấy ngay mức độ lệch phân bố giữa dữ liệu huấn luyện và `test_sample`.


In [2]:
# Auto-discover the correct data path
_candidates = [
    '/kaggle/input/semeval-2026-task-13-subtask-a',
    '/kaggle/input/semeval-2026-task-13-subtask-a/Task_A',
    '/kaggle/input/sem-eval-2026-task-13-subtask-a/Task_A',
]
DATA_DIR = None
for _c in _candidates:
    if os.path.exists(os.path.join(_c, 'train.parquet')):
        DATA_DIR = _c
        break

if DATA_DIR is None:
    _found = glob.glob('/kaggle/input/**/train.parquet', recursive=True)
    if _found:
        DATA_DIR = os.path.dirname(_found[0])

if DATA_DIR is None:
    raise FileNotFoundError('Could not locate train.parquet under /kaggle/input')

print(f'DATA_DIR: {DATA_DIR}')
print('Files:', os.listdir(DATA_DIR))

train       = pd.read_parquet(f'{DATA_DIR}/train.parquet')
val         = pd.read_parquet(f'{DATA_DIR}/validation.parquet')
test        = pd.read_parquet(f'{DATA_DIR}/test.parquet')
test_sample = pd.read_parquet(f'{DATA_DIR}/test_sample.parquet')

for df in [train, val, test, test_sample]:
    df['code'] = df['code'].fillna('')

print(f'Train: {train.shape}, Val: {val.shape}, Test: {test.shape}, test_sample: {test_sample.shape}')
print(f'Train label dist: {train["label"].value_counts().to_dict()}')
print(f'Val label dist:   {val["label"].value_counts().to_dict()}')
print(f'test_sample dist: {test_sample["label"].value_counts().to_dict()}')

train_prior_ai = float(train['label'].mean())
val_prior_ai   = float(val['label'].mean())
test_prior_ai  = float(test_sample['label'].mean())

print(f'\nTrain prior P(AI): {train_prior_ai:.3f}')
print(f'Val prior   P(AI): {val_prior_ai:.3f}')
print(f'Test prior  P(AI): {test_prior_ai:.3f}  <-- target shift proxy')


DATA_DIR: /kaggle/input/competitions/sem-eval-2026-task-13-subtask-a/Task_A
Files: ['sample_submission.csv', 'train.parquet', 'test_sample.parquet', 'validation.parquet', 'test.parquet']
Train: (500000, 4), Val: (100000, 4), Test: (500000, 2), test_sample: (1000, 4)
Train label dist: {1: 261525, 0: 238475}
Val label dist:   {1: 52305, 0: 47695}
test_sample dist: {0: 777, 1: 223}

Train prior P(AI): 0.523
Val prior   P(AI): 0.523
Test prior  P(AI): 0.223  <-- target shift proxy


### Hình cho report: quy mô dữ liệu và tỷ lệ nhãn

Tôi dùng hình này để tóm tắt nhanh kích thước từng split và mức độ lệch nhãn giữa các tập. Đây là chỗ giúp tôi thấy rõ `test_sample` có prior khác đáng kể so với `train` và `validation`.


In [ ]:
import matplotlib.pyplot as plt

split_overview = pd.DataFrame({
    'split': ['train', 'validation', 'test_sample'],
    'n_samples': [len(train), len(val), len(test_sample)],
    'ai_rate': [train_prior_ai, val_prior_ai, test_prior_ai]
})

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].bar(split_overview['split'], split_overview['n_samples'])
axes[0].set_title('Quy mô từng split')
axes[0].set_ylabel('Số mẫu')
for i, v in enumerate(split_overview['n_samples']):
    axes[0].text(i, v, f'{v:,}', ha='center', va='bottom')

axes[1].bar(split_overview['split'], split_overview['ai_rate'])
axes[1].set_title('Tỷ lệ nhãn AI theo split')
axes[1].set_ylabel('P(label = AI)')
axes[1].set_ylim(0, 1)
for i, v in enumerate(split_overview['ai_rate']):
    axes[1].text(i, v, f'{v:.3f}', ha='center', va='bottom')

plt.tight_layout()
plt.show()


## 2. Trích xuất handcrafted features giàu thông tin

Ngoài TF-IDF, tôi chủ động xây thêm một nhóm đặc trưng cấu trúc để mô tả code ở mức style và định dạng. Nhóm này được thiết kế để khác với bag-of-ngrams càng nhiều càng tốt, vì tôi muốn nhánh tree học từ một không gian tín hiệu có bản chất khác.

Các nhóm feature chính mà tôi dùng gồm:
- cấu trúc dòng và indentation
- comment và docstring
- kiểu đặt tên biến / identifier
- dấu hiệu ngôn ngữ lập trình
- thống kê lặp lại, formatting và mật độ ký tự

Tôi dùng bộ feature này theo hai cách:
1. đưa bản đã scale vào các mô hình sparse tuyến tính  
2. giữ bản raw cho nhánh tree-based


In [3]:
FEAT_NAMES = [
    'code_len', 'log_len', 'n_lines', 'n_non_empty', 'n_blank',
    'non_empty_ratio', 'blank_ratio',
    'avg_line_len', 'std_line_len', 'max_line_len', 'median_line_len', 'chars_per_nonempty',
    'n_tokens', 'unique_tokens', 'unique_ratio', 'avg_token_len',
    'n_identifiers', 'unique_identifiers', 'ident_unique_ratio',
    'snake_case', 'camel_case', 'upper_ident',
    'snake_ratio', 'camel_ratio', 'upper_ident_ratio',
    'n_numbers', 'alpha_ratio', 'digit_ratio', 'space_ratio', 'punct_ratio',
    'comment_lines', 'inline_comment_lines', 'n_block_comments', 'n_docstrings',
    'comment_ratio', 'inline_comment_ratio',
    'block_comment_chars', 'docstring_chars', 'comment_char_ratio',
    'avg_indent', 'max_indent', 'std_indent', 'median_indent',
    'tab_indent_ratio', 'space_indent_ratio', 'trailing_ws_ratio',
    'repeated_line_ratio', 'max_token_freq_ratio',
    'english_words', 'english_ratio',
    'operators', 'op_density',
    'semicolons', 'commas', 'colons', 'dots',
    'n_string_literals', 'avg_string_len',
    'if_count', 'else_count', 'for_count', 'while_count',
    'return_count', 'import_count', 'class_count', 'def_count',
    'try_count', 'except_count', 'lambda_count', 'async_count', 'await_count',
    'c_like_score', 'python_score', 'js_score', 'java_score', 'sql_score'
]

assert len(FEAT_NAMES) == 76

def _process_single(text):
    if not isinstance(text, str):
        text = ''
    if len(text) == 0:
        return [0.0] * len(FEAT_NAMES)

    lines = text.split('\n')
    n_lines = len(lines)
    non_empty = [l for l in lines if l.strip()]
    n_non_empty = len(non_empty)
    n_blank = n_lines - n_non_empty

    line_lens = [len(l) for l in lines] if lines else [0]
    code_len = len(text)
    avg_line_len = float(np.mean(line_lens))
    std_line_len = float(np.std(line_lens))
    max_line_len = float(max(line_lens))
    median_line_len = float(np.median(line_lens))
    chars_per_nonempty = code_len / max(n_non_empty, 1)

    all_tokens = re.findall(r'\w+', text)
    n_tokens = len(all_tokens)
    unique_tokens = len(set(all_tokens))
    unique_ratio = unique_tokens / max(n_tokens, 1)
    avg_token_len = float(np.mean([len(t) for t in all_tokens])) if all_tokens else 0.0

    identifiers = re.findall(r'[A-Za-z_][A-Za-z0-9_]*', text)
    n_identifiers = len(identifiers)
    unique_identifiers = len(set(identifiers))
    ident_unique_ratio = unique_identifiers / max(n_identifiers, 1)

    snake_case = sum(1 for t in identifiers if '_' in t and t.lower() == t)
    camel_case = sum(1 for t in identifiers if re.search(r'[a-z][A-Z]', t) is not None)
    upper_ident = sum(1 for t in identifiers if len(t) >= 2 and t.upper() == t and any(c.isalpha() for c in t))

    n_numbers = len(re.findall(r'\b\d+(?:\.\d+)?\b', text))
    alpha_chars = sum(ch.isalpha() for ch in text)
    digit_chars = sum(ch.isdigit() for ch in text)
    space_chars = sum(ch.isspace() for ch in text)
    punct_chars = sum((not ch.isalnum()) and (not ch.isspace()) for ch in text)

    comment_lines = sum(1 for l in lines if l.strip().startswith(('#', '//', '--')))
    inline_comment_lines = sum(
        1 for l in lines
        if ('#' in l and not l.strip().startswith('#'))
        or ('//' in l and not l.strip().startswith('//'))
    )
    block_comments = re.findall(r'/\*[\s\S]*?\*/', text)
    block_comment_chars = float(sum(len(b) for b in block_comments))

    docstrings = re.findall(r'"""[\s\S]*?"""|\'{3}[\s\S]*?\'{3}', text)
    docstring_chars = float(sum(len(d) for d in docstrings))

    comment_char_ratio = (
        sum(len(l) for l in lines if l.strip().startswith(('#', '//', '--'))) + block_comment_chars + docstring_chars
    ) / max(code_len, 1)

    if non_empty:
        indents = [len(l) - len(l.lstrip(' \t')) for l in non_empty]
        avg_indent = float(np.mean(indents))
        max_indent = float(max(indents))
        std_indent = float(np.std(indents))
        median_indent = float(np.median(indents))
        tab_indent_lines = sum(1 for l in non_empty if l.startswith('\t'))
        space_indent_lines = sum(1 for l in non_empty if l.startswith(' '))
    else:
        avg_indent = max_indent = std_indent = median_indent = 0.0
        tab_indent_lines = space_indent_lines = 0

    trailing_ws_ratio = sum(1 for l in lines if l != l.rstrip()) / max(n_lines, 1)
    repeated_line_ratio = (n_non_empty - len(set(non_empty))) / max(n_non_empty, 1)

    token_counts = Counter(all_tokens)
    max_token_freq_ratio = (max(token_counts.values()) / max(n_tokens, 1)) if token_counts else 0.0

    english_words = len(re.findall(
        r'\b(?:the|is|are|this|that|for|with|returns?|param|function|method|class|'
        r'object|value|result|output|input|data|list|string|number|type|if|else|'
        r'while|loop|index|array|dict|key|true|false|none|null|void|bool|int|float|'
        r'str|char|size|length|count|name|error|exception|check|valid|create|update|'
        r'delete|get|set|add|remove|find|sort|compare|convert|print|read|write|open|'
        r'close|file|path|directory|module|package|library|test|main|init|run)\b',
        text.lower()
    ))

    operators = len(re.findall(r'[+\-*/=<>!&|^~%]{1,3}', text))
    string_literals = re.findall(r'"[^"\n]{0,200}"|\'[^\'\n]{0,200}\'', text)
    avg_string_len = float(np.mean([len(s) for s in string_literals])) if string_literals else 0.0

    kw = {
        'if': len(re.findall(r'\bif\b', text)),
        'else': len(re.findall(r'\belse\b', text)),
        'for': len(re.findall(r'\bfor\b', text)),
        'while': len(re.findall(r'\bwhile\b', text)),
        'return': len(re.findall(r'\breturn\b', text)),
        'import': len(re.findall(r'\bimport\b', text)),
        'class': len(re.findall(r'\bclass\b', text)),
        'def': len(re.findall(r'\bdef\b', text)),
        'try': len(re.findall(r'\btry\b', text)),
        'except': len(re.findall(r'\bexcept\b', text)),
        'lambda': len(re.findall(r'\blambda\b', text)),
        'async': len(re.findall(r'\basync\b', text)),
        'await': len(re.findall(r'\bawait\b', text)),
    }

    c_like_score = int('#include' in text) + text.count('->') + text.count(';') + text.count('{')
    python_score = kw['def'] + kw['import'] + text.count('from ') + text.count('"""')
    js_score = len(re.findall(r'\b(?:function|const|let|var)\b', text)) + text.count('=>') + text.count('console.log')
    java_score = int('public static void main' in text) + text.count('System.out') + kw['class']
    sql_score = len(re.findall(r'\b(?:select|from|where|join|insert|update|delete)\b', text.lower()))

    return [
        code_len, float(np.log1p(code_len)), n_lines, n_non_empty, n_blank,
        n_non_empty / max(n_lines, 1), n_blank / max(n_lines, 1),
        avg_line_len, std_line_len, max_line_len, median_line_len, chars_per_nonempty,
        n_tokens, unique_tokens, unique_ratio, avg_token_len,
        n_identifiers, unique_identifiers, ident_unique_ratio,
        snake_case, camel_case, upper_ident,
        snake_case / max(n_identifiers, 1), camel_case / max(n_identifiers, 1), upper_ident / max(n_identifiers, 1),
        n_numbers, alpha_chars / max(code_len, 1), digit_chars / max(code_len, 1),
        space_chars / max(code_len, 1), punct_chars / max(code_len, 1),
        comment_lines, inline_comment_lines, len(block_comments), len(docstrings),
        comment_lines / max(n_lines, 1), inline_comment_lines / max(n_lines, 1),
        block_comment_chars, docstring_chars, comment_char_ratio,
        avg_indent, max_indent, std_indent, median_indent,
        tab_indent_lines / max(n_non_empty, 1), space_indent_lines / max(n_non_empty, 1), trailing_ws_ratio,
        repeated_line_ratio, max_token_freq_ratio,
        english_words, english_words / max(n_tokens, 1),
        operators, operators / max(n_tokens, 1),
        text.count(';'), text.count(','), text.count(':'), text.count('.'),
        len(string_literals), avg_string_len,
        kw['if'], kw['else'], kw['for'], kw['while'],
        kw['return'], kw['import'], kw['class'], kw['def'],
        kw['try'], kw['except'], kw['lambda'], kw['async'], kw['await'],
        c_like_score, python_score, js_score, java_score, sql_score
    ]

def extract_features(df, n_jobs=4):
    rows = Parallel(n_jobs=n_jobs, backend='loky')(
        delayed(_process_single)(text) for text in df['code']
    )
    return pd.DataFrame(rows, columns=FEAT_NAMES).astype(np.float32)

print(f'Feature function defined ({len(FEAT_NAMES)} features)')

print('Extracting train/val/test/test_sample handcrafted features...')
train_feats_raw = extract_features(train)
val_feats_raw   = extract_features(val)
test_feats_raw  = extract_features(test)
ts_feats_raw    = extract_features(test_sample)

print('Shapes:')
print('  train:', train_feats_raw.shape)
print('  val:  ', val_feats_raw.shape)
print('  test: ', test_feats_raw.shape)
print('  ts:   ', ts_feats_raw.shape)

print('\nTop variance features:')
display(train_feats_raw.var().sort_values(ascending=False).head(12).to_frame('variance'))


Feature function defined (76 features)
Extracting train/val/test/test_sample handcrafted features...
Shapes:
  train: (500000, 76)
  val:   (100000, 76)
  test:  (500000, 76)
  ts:    (1000, 76)

Top variance features:


,variance
code_len,1.884219e+06
max_line_len,5.181013e+05
docstring_chars,3.583632e+04
n_tokens,2.898151e+04
std_line_len,2.586580e+04
n_identifiers,2.156823e+04
block_comment_chars,2.134781e+04
chars_per_nonempty,4.978528e+03
n_lines,4.024146e+03
avg_line_len,3.146475e+03


### Hình cho report: phân bố một số handcrafted features theo nhãn

Ở đây tôi chọn vài đặc trưng dễ diễn giải để minh họa sự khác biệt giữa code Human và AI trong tập huấn luyện. Tôi ưu tiên các feature vừa có ý nghĩa trực quan, vừa liên quan đến style và cấu trúc mã nguồn.


In [ ]:
import matplotlib.pyplot as plt

feature_cols = ['log_len', 'comment_ratio', 'unique_ratio', 'avg_indent']
feature_titles = {
    'log_len': 'Độ dài code (log scale)',
    'comment_ratio': 'Tỷ lệ comment',
    'unique_ratio': 'Tỷ lệ token unique',
    'avg_indent': 'Indentation trung bình'
}

plot_df = train_feats_raw.copy()
plot_df['label'] = train['label'].values

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

for ax, col in zip(axes, feature_cols):
    x_h = plot_df.loc[plot_df['label'] == 0, col].astype(float).values
    x_ai = plot_df.loc[plot_df['label'] == 1, col].astype(float).values

    hi = np.nanpercentile(np.concatenate([x_h, x_ai]), 99)
    bins = np.linspace(0, hi if hi > 0 else 1, 40)

    ax.hist(np.clip(x_h, 0, hi), bins=bins, alpha=0.6, density=True, label='Human')
    ax.hist(np.clip(x_ai, 0, hi), bins=bins, alpha=0.6, density=True, label='AI')
    ax.set_title(feature_titles[col])
    ax.set_xlabel(col)
    ax.set_ylabel('Mật độ')
    ax.legend()

plt.tight_layout()
plt.show()


## 3. TF-IDF trên code tokens

Ở nhánh văn bản, tôi kết hợp hai góc nhìn:
- **word/token TF-IDF** để giữ các token có nghĩa ở mức từ vựng
- **char TF-IDF** để bắt các mẫu ngắn ở mức ký tự, rất hữu ích với cú pháp, dấu câu và style code

Cách làm này thường cho độ phủ tốt hơn so với chỉ dùng một loại biểu diễn.


In [4]:
def code_tokenizer(text):
    return re.findall(r'[a-zA-Z_][a-zA-Z0-9_]*|\d+|[+\-*/=<>!&|^~%]+|[{}()\[\];,.:]+', text)

print('Fitting word TF-IDF...')
tfidf_word = TfidfVectorizer(
    tokenizer=code_tokenizer,
    max_features=20000,
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)
train_tfidf_w = tfidf_word.fit_transform(train['code'])
val_tfidf_w   = tfidf_word.transform(val['code'])
test_tfidf_w  = tfidf_word.transform(test['code'])
ts_tfidf_w    = tfidf_word.transform(test_sample['code'])
print(f'Word TF-IDF shape: {train_tfidf_w.shape}')

print('Fitting char TF-IDF...')
tfidf_char = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(2, 4),
    max_features=12000,
    min_df=5,
    max_df=0.95,
    sublinear_tf=True,
    dtype=np.float32
)
train_tfidf_c = tfidf_char.fit_transform(train['code'])
val_tfidf_c   = tfidf_char.transform(val['code'])
test_tfidf_c  = tfidf_char.transform(test['code'])
ts_tfidf_c    = tfidf_char.transform(test_sample['code'])
print(f'Char TF-IDF shape: {train_tfidf_c.shape}')


Fitting word TF-IDF...
Word TF-IDF shape: (500000, 20000)
Fitting char TF-IDF...
Char TF-IDF shape: (500000, 12000)


### Hình cho report: kích thước không gian đặc trưng

Hình này giúp tôi mô tả rõ mỗi nhánh linear đang học trên bao nhiêu chiều đặc trưng, đồng thời cho thấy phần TF-IDF vẫn chiếm phần lớn dung lượng biểu diễn.


In [ ]:
import matplotlib.pyplot as plt

feature_space = pd.DataFrame({
    'block': ['handcrafted', 'word_tfidf', 'char_tfidf', 'text_total', 'linear_total'],
    'n_features': [
        len(FEAT_NAMES),
        train_tfidf_w.shape[1],
        train_tfidf_c.shape[1],
        train_tfidf_w.shape[1] + train_tfidf_c.shape[1],
        len(FEAT_NAMES) + train_tfidf_w.shape[1] + train_tfidf_c.shape[1]
    ]
})

plt.figure(figsize=(9, 4))
plt.bar(feature_space['block'], feature_space['n_features'])
plt.title('Kích thước từng khối đặc trưng')
plt.ylabel('Số chiều')
plt.xticks(rotation=20)
for i, v in enumerate(feature_space['n_features']):
    plt.text(i, v, f'{v:,}', ha='center', va='bottom')
plt.tight_layout()
plt.show()


## 4. Tạo đầu vào cho từng nhánh mô hình

Tại đây tôi ghép các khối đặc trưng theo đúng vai trò của từng nhánh:
- các nhánh linear dùng **handcrafted features đã scale + sparse TF-IDF**
- nhánh tree chỉ dùng **handcrafted features bản raw**

Tôi tách như vậy để vừa giữ được tính ổn định cho mô hình tuyến tính, vừa không làm mất lợi thế của tree model trên các đặc trưng dense.


In [5]:
y_train = train['label'].values.astype(np.int32)
y_val   = val['label'].values.astype(np.int32)
y_ts    = test_sample['label'].values.astype(np.int32)

scaler = StandardScaler()
train_struct_scaled = scaler.fit_transform(train_feats_raw.values)
val_struct_scaled   = scaler.transform(val_feats_raw.values)
test_struct_scaled  = scaler.transform(test_feats_raw.values)
ts_struct_scaled    = scaler.transform(ts_feats_raw.values)

X_train_struct = csr_matrix(train_struct_scaled.astype(np.float32))
X_val_struct   = csr_matrix(val_struct_scaled.astype(np.float32))
X_test_struct  = csr_matrix(test_struct_scaled.astype(np.float32))
X_ts_struct    = csr_matrix(ts_struct_scaled.astype(np.float32))

X_train_text = hstack([train_tfidf_w, train_tfidf_c], format='csr')
X_val_text   = hstack([val_tfidf_w,   val_tfidf_c],   format='csr')
X_test_text  = hstack([test_tfidf_w,  test_tfidf_c],  format='csr')
X_ts_text    = hstack([ts_tfidf_w,    ts_tfidf_c],    format='csr')

print(f'Structural features: {X_train_struct.shape[1]}')
print(f'Text features:       {X_train_text.shape[1]}')
print(f'Total linear dims:   {X_train_struct.shape[1] + X_train_text.shape[1]}')

del train_tfidf_w, train_tfidf_c, val_tfidf_w, val_tfidf_c
del test_tfidf_w, test_tfidf_c, ts_tfidf_w, ts_tfidf_c
gc.collect()


Structural features: 76
Text features:       32000
Total linear dims:   32076


54

## 5. Nhánh sparse linear baseline

Đây là baseline tuyến tính chính của tôi. Mô hình này học trên toàn bộ không gian sparse gồm structural features đã scale và TF-IDF, nên thường bắt rất tốt các tín hiệu từ token, cú pháp và pattern xuất hiện lặp lại trong code.


In [6]:
base_clf = SGDClassifier(
    loss='log_loss',
    alpha=1e-5,
    class_weight='balanced',
    max_iter=3000,
    random_state=42
)

print('Training base sparse logistic model...')
t0 = time.time()
X_train_base = hstack([X_train_struct, X_train_text], format='csr')
base_clf.fit(X_train_base, y_train)
print(f'Base training done in {(time.time()-t0)/60:.1f} min')

X_val_base = hstack([X_val_struct, X_val_text], format='csr')
X_ts_base  = hstack([X_ts_struct,  X_ts_text],  format='csr')

base_val_probs = base_clf.predict_proba(X_val_base)[:, 1].astype(np.float32)
base_ts_probs  = base_clf.predict_proba(X_ts_base)[:, 1].astype(np.float32)

del X_train_base, X_val_base, X_ts_base
gc.collect()


Training base sparse logistic model...
Base training done in 0.7 min


0

## 6. Nhánh NB-SVM

Ở nhánh này, tôi giữ nguyên khung mô hình tuyến tính nhưng reweight phần text bằng log-count ratio theo tinh thần NB-SVM. Ý tưởng của tôi là làm nổi bật những token có tính phân biệt lớp cao hơn trước khi đưa vào bộ phân loại logistic.


In [7]:
def compute_log_count_ratio(X, y, alpha=1.0):
    pos = X[y == 1]
    neg = X[y == 0]

    p = np.asarray(pos.sum(axis=0)).ravel().astype(np.float64) + alpha
    q = np.asarray(neg.sum(axis=0)).ravel().astype(np.float64) + alpha

    p /= p.sum()
    q /= q.sum()
    r = np.log(p) - np.log(q)
    return r.astype(np.float32)

r_text = compute_log_count_ratio(X_train_text, y_train, alpha=1.0)
print('Computed NB log-count-ratio vector:', r_text.shape)
print('ratio stats:', float(r_text.min()), float(r_text.mean()), float(r_text.max()))

X_train_nb = hstack([X_train_struct, X_train_text.multiply(r_text)], format='csr')
X_val_nb   = hstack([X_val_struct,   X_val_text.multiply(r_text)],   format='csr')
X_ts_nb    = hstack([X_ts_struct,    X_ts_text.multiply(r_text)],    format='csr')

nbsvm_clf = SGDClassifier(
    loss='log_loss',
    alpha=5e-6,
    class_weight='balanced',
    max_iter=3000,
    random_state=42
)

print('Training NB-SVM-style linear model...')
t0 = time.time()
nbsvm_clf.fit(X_train_nb, y_train)
print(f'NB-SVM training done in {(time.time()-t0)/60:.1f} min')

nb_val_probs = nbsvm_clf.predict_proba(X_val_nb)[:, 1].astype(np.float32)
nb_ts_probs  = nbsvm_clf.predict_proba(X_ts_nb)[:, 1].astype(np.float32)

del X_train_nb, X_val_nb, X_ts_nb
gc.collect()


Computed NB log-count-ratio vector: (32000,)
ratio stats: -4.632547378540039 0.5843214988708496 6.702520370483398
Training NB-SVM-style linear model...
NB-SVM training done in 0.8 min


0

## 7. Nhánh tree trên handcrafted features

Nhánh này chỉ nhìn vào handcrafted features và được dùng như một nguồn tín hiệu bổ sung cho hai nhánh sparse. Kỳ vọng của tôi là tree model sẽ học được các tương tác phi tuyến giữa những đặc trưng style mà mô hình tuyến tính khó tận dụng hết.


In [8]:
def make_tree_model(n_estimators=500, random_state=42):
    if HAVE_LGBM:
        return LGBMClassifier(
            objective='binary',
            n_estimators=n_estimators,
            learning_rate=0.03,
            num_leaves=31,
            min_child_samples=80,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_alpha=0.25,
            reg_lambda=2.0,
            random_state=random_state,
            n_jobs=-1,
            verbosity=-1
        )
    return HistGradientBoostingClassifier(
        loss='log_loss',
        learning_rate=0.03,
        max_iter=n_estimators,
        max_leaf_nodes=31,
        min_samples_leaf=80,
        random_state=random_state
    )

tree_clf = make_tree_model(n_estimators=500, random_state=42)

print('Training tree model on handcrafted features...')
t0 = time.time()
if HAVE_LGBM:
    tree_clf.fit(
        train_feats_raw, y_train,
        eval_set=[(val_feats_raw, y_val)],
        eval_metric='binary_logloss',
        callbacks=[lgb.early_stopping(50, verbose=False)]
    )
    tree_best_round = int(tree_clf.best_iteration_ or tree_clf.n_estimators)
else:
    tree_clf.fit(train_feats_raw, y_train)
    tree_best_round = 500

print(f'Tree training done in {(time.time()-t0)/60:.1f} min')
print('tree_best_round =', tree_best_round)

tree_val_probs = tree_clf.predict_proba(val_feats_raw)[:, 1].astype(np.float32)
tree_ts_probs  = tree_clf.predict_proba(ts_feats_raw)[:, 1].astype(np.float32)

if HAVE_LGBM:
    fi = pd.Series(tree_clf.feature_importances_, index=FEAT_NAMES).sort_values(ascending=False).head(20)
    print('\nTop handcrafted feature importances:')
    display(fi.to_frame('importance'))


Training tree model on handcrafted features...
Tree training done in 0.7 min
tree_best_round = 500

Top handcrafted feature importances:


,importance
space_indent_ratio,861
non_empty_ratio,631
n_blank,546
comment_char_ratio,536
semicolons,525
trailing_ws_ratio,511
punct_ratio,449
class_count,404
return_count,381
median_line_len,371


### Hình cho report: top handcrafted features của nhánh tree

Tôi dùng hình này để xem nhanh những đặc trưng cấu trúc nào đang được nhánh tree khai thác mạnh nhất. Nếu chạy bằng LightGBM thì biểu đồ này thường khá hữu ích để diễn giải mô hình.


In [ ]:
import matplotlib.pyplot as plt

if hasattr(tree_clf, 'feature_importances_'):
    fi_plot = pd.Series(tree_clf.feature_importances_, index=FEAT_NAMES).sort_values(ascending=True).tail(20)

    plt.figure(figsize=(9, 7))
    plt.barh(fi_plot.index, fi_plot.values)
    plt.title('Top 20 handcrafted feature importances (tree branch)')
    plt.xlabel('Importance')
    plt.tight_layout()
    plt.show()
else:
    print('Model tree hiện tại không cung cấp feature_importances_; bỏ qua biểu đồ này.')


## 8. So sánh từng nhánh đơn lẻ và các phương án blend

Ở bước này tôi đánh giá:
- mô hình sparse logistic baseline
- nhánh NB-SVM
- nhánh tree trên handcrafted features
- các tổ hợp convex blend của ba nhánh

Quy tắc chọn cấu hình:
1. ưu tiên macro F1 trên `test_sample`  
2. với các ứng viên rất sát nhau trên `test_sample`, tôi dùng thêm chất lượng trên validation để ưu tiên phương án ổn định hơn


In [9]:
def prior_correct(probs, train_prior_ai, test_prior_ai):
    probs = np.clip(probs, 1e-6, 1 - 1e-6)
    odds = probs / (1.0 - probs)
    ratio = (test_prior_ai / (1.0 - test_prior_ai)) / (train_prior_ai / (1.0 - train_prior_ai))
    odds_corr = odds * ratio
    return odds_corr / (1.0 + odds_corr)

def tune_threshold(y_true, probs, grid):
    best_f1, best_t = -1.0, 0.5
    for t in grid:
        pred = (probs > t).astype(np.int32)
        f1 = f1_score(y_true, pred, average='macro')
        if f1 > best_f1:
            best_f1, best_t = float(f1), float(t)
    return best_f1, best_t

component_probs = {
    'base': {'val': base_val_probs, 'ts': base_ts_probs},
    'nbsvm': {'val': nb_val_probs, 'ts': nb_ts_probs},
    'tree': {'val': tree_val_probs, 'ts': tree_ts_probs},
}

blend_specs = []
for bw in np.arange(0.0, 1.01, 0.1):
    for nw in np.arange(0.0, 1.01 - bw, 0.1):
        tw = round(1.0 - bw - nw, 10)
        if tw < -1e-9:
            continue
        bw_r = round(float(bw), 1)
        nw_r = round(float(nw), 1)
        tw_r = round(float(max(tw, 0.0)), 1)
        if bw_r == 0 and nw_r == 0 and tw_r == 0:
            continue
        blend_specs.append((bw_r, nw_r, tw_r))

# de-duplicate small floating-point artifacts
blend_specs = sorted(set(blend_specs))

rows = []
ts_grid_direct = np.arange(0.10, 0.90, 0.01)
ts_grid_corr   = np.arange(0.05, 0.95, 0.005)

for bw, nw, tw in blend_specs:
    val_probs = (
        bw * component_probs['base']['val'] +
        nw * component_probs['nbsvm']['val'] +
        tw * component_probs['tree']['val']
    ).astype(np.float32)

    ts_probs = (
        bw * component_probs['base']['ts'] +
        nw * component_probs['nbsvm']['ts'] +
        tw * component_probs['tree']['ts']
    ).astype(np.float32)

    direct_f1, direct_t = tune_threshold(y_ts, ts_probs, ts_grid_direct)
    ts_probs_corr = prior_correct(ts_probs, train_prior_ai, test_prior_ai)
    corr_f1, corr_t = tune_threshold(y_ts, ts_probs_corr, ts_grid_corr)

    if corr_f1 >= direct_f1:
        chosen_strategy = 'prior_corrected'
        chosen_f1 = corr_f1
        chosen_thresh = corr_t
    else:
        chosen_strategy = 'direct_threshold'
        chosen_f1 = direct_f1
        chosen_thresh = direct_t

    # Stability score from validation, tuned independently on val.
    val_direct_best, _ = tune_threshold(y_val, val_probs, ts_grid_direct)
    val_corr_best, _   = tune_threshold(y_val, prior_correct(val_probs, train_prior_ai, test_prior_ai), ts_grid_corr)
    val_best_f1 = max(val_direct_best, val_corr_best)

    active_models = int((bw > 0) + (nw > 0) + (tw > 0))

    if active_models == 1:
        if bw == 1.0:
            name = 'base'
        elif nw == 1.0:
            name = 'nbsvm'
        else:
            name = 'tree'
    else:
        name = f'blend_b{bw:.1f}_n{nw:.1f}_t{tw:.1f}'

    rows.append({
        'candidate': name,
        'base_w': bw,
        'nb_w': nw,
        'tree_w': tw,
        'n_active_models': active_models,
        'direct_f1': direct_f1,
        'direct_thresh': direct_t,
        'corr_f1': corr_f1,
        'corr_thresh': corr_t,
        'chosen_strategy': chosen_strategy,
        'chosen_f1': chosen_f1,
        'chosen_thresh': chosen_thresh,
        'val_best_f1': val_best_f1,
    })

leaderboard = pd.DataFrame(rows).sort_values(
    ['chosen_f1', 'val_best_f1', 'n_active_models'],
    ascending=[False, False, True]
).reset_index(drop=True)

display(leaderboard.head(20))

top_ts = float(leaderboard['chosen_f1'].max())
stable_pool = leaderboard[leaderboard['chosen_f1'] >= top_ts - 0.0025].copy()
stable_pool = stable_pool.sort_values(
    ['val_best_f1', 'chosen_f1', 'n_active_models'],
    ascending=[False, False, True]
).reset_index(drop=True)

print('Stable pool (within 0.0025 of best test_sample F1):')
display(stable_pool.head(12))

best = stable_pool.iloc[0].to_dict()
best_name = best['candidate']
best_strategy = best['chosen_strategy']
best_thresh = float(best['chosen_thresh'])
best_f1 = float(best['chosen_f1'])
best_base_w = float(best['base_w'])
best_nb_w = float(best['nb_w'])
best_tree_w = float(best['tree_w'])

print('\nSelected candidate:')
print(best)

best_ts_probs = (
    best_base_w * base_ts_probs +
    best_nb_w   * nb_ts_probs +
    best_tree_w * tree_ts_probs
).astype(np.float32)

if best_strategy == 'prior_corrected':
    best_ts_probs_eval = prior_correct(best_ts_probs, train_prior_ai, test_prior_ai)
else:
    best_ts_probs_eval = best_ts_probs

best_ts_preds = (best_ts_probs_eval > best_thresh).astype(np.int32)
print('\nSelected candidate report on test_sample:')
print(classification_report(y_ts, best_ts_preds, target_names=['Human', 'AI']))


,candidate,base_w,nb_w,tree_w,n_active_models,direct_f1,direct_thresh,corr_f1,corr_thresh,chosen_strategy,chosen_f1,chosen_thresh,val_best_f1
0,blend_b0.4_n0.3_t0.3,0.4,0.3,0.3,3,0.466078,0.89,0.569822,0.945,prior_corrected,0.569822,0.945,0.980355
1,blend_b0.3_n0.4_t0.3,0.3,0.4,0.3,3,0.464438,0.89,0.568142,0.945,prior_corrected,0.568142,0.945,0.980200
2,blend_b0.7_n0.2_t0.1,0.7,0.2,0.1,3,0.451574,0.89,0.568142,0.945,prior_corrected,0.568142,0.945,0.973429
3,blend_b0.4_n0.5_t0.1,0.4,0.5,0.1,3,0.453035,0.89,0.567883,0.945,prior_corrected,0.567883,0.945,0.975098
4,blend_b0.5_n0.4_t0.1,0.5,0.4,0.1,3,0.453990,0.89,0.567390,0.945,prior_corrected,0.567390,0.945,0.975119
5,blend_b0.3_n0.3_t0.4,0.3,0.3,0.4,3,0.465263,0.89,0.567083,0.945,prior_corrected,0.567083,0.945,0.983887
6,blend_b0.2_n0.4_t0.4,0.2,0.4,0.4,3,0.458647,0.89,0.566497,0.945,prior_corrected,0.566497,0.945,0.983787
7,blend_b0.4_n0.4_t0.2,0.4,0.4,0.2,3,0.460541,0.89,0.565370,0.945,prior_corrected,0.565370,0.945,0.977855
8,blend_b0.2_n0.5_t0.3,0.2,0.5,0.3,3,0.457802,0.89,0.565057,0.945,prior_corrected,0.565057,0.945,0.979726
9,blend_b0.6_n0.1_t0.3,0.6,0.1,0.3,3,0.466078,0.89,0.564344,0.945,prior_corrected,0.564344,0.945,0.978756


Stable pool (within 0.0025 of best test_sample F1):


,candidate,base_w,nb_w,tree_w,n_active_models,direct_f1,direct_thresh,corr_f1,corr_thresh,chosen_strategy,chosen_f1,chosen_thresh,val_best_f1
0,blend_b0.4_n0.3_t0.3,0.4,0.3,0.3,3,0.466078,0.89,0.569822,0.945,prior_corrected,0.569822,0.945,0.980355
1,blend_b0.3_n0.4_t0.3,0.3,0.4,0.3,3,0.464438,0.89,0.568142,0.945,prior_corrected,0.568142,0.945,0.980200
2,blend_b0.5_n0.4_t0.1,0.5,0.4,0.1,3,0.453990,0.89,0.567390,0.945,prior_corrected,0.567390,0.945,0.975119
3,blend_b0.4_n0.5_t0.1,0.4,0.5,0.1,3,0.453035,0.89,0.567883,0.945,prior_corrected,0.567883,0.945,0.975098
4,blend_b0.7_n0.2_t0.1,0.7,0.2,0.1,3,0.451574,0.89,0.568142,0.945,prior_corrected,0.568142,0.945,0.973429



Selected candidate:
{'candidate': 'blend_b0.4_n0.3_t0.3', 'base_w': 0.4, 'nb_w': 0.3, 'tree_w': 0.3, 'n_active_models': 3, 'direct_f1': 0.4660778803693296, 'direct_thresh': 0.8899999999999996, 'corr_f1': 0.569822032973032, 'corr_thresh': 0.9449999999999996, 'chosen_strategy': 'prior_corrected', 'chosen_f1': 0.569822032973032, 'chosen_thresh': 0.9449999999999996, 'val_best_f1': 0.9803550015673481}

Selected candidate report on test_sample:
              precision    recall  f1-score   support

       Human       0.87      0.58      0.69       777
          AI       0.32      0.71      0.45       223

    accuracy                           0.61      1000
   macro avg       0.60      0.64      0.57      1000
weighted avg       0.75      0.61      0.64      1000



### Hình cho report: mặt bằng các ứng viên blend

Tôi dùng biểu đồ này để nhìn tương quan giữa chất lượng trên validation và `test_sample` của toàn bộ ứng viên. Điểm được khoanh là cấu hình tôi chọn sau khi áp dụng quy tắc chọn blend ở trên.


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 6))
plt.scatter(leaderboard['val_best_f1'], leaderboard['chosen_f1'], alpha=0.5)

single_mask = leaderboard['candidate'].isin(['base', 'nbsvm', 'tree'])
plt.scatter(
    leaderboard.loc[single_mask, 'val_best_f1'],
    leaderboard.loc[single_mask, 'chosen_f1'],
    s=80,
    label='Single models'
)

best_row = leaderboard.iloc[0]
plt.scatter([best_row['val_best_f1']], [best_row['chosen_f1']], s=150, marker='*', label='Selected candidate')
plt.annotate(
    best_row['candidate'],
    (best_row['val_best_f1'], best_row['chosen_f1']),
    textcoords='offset points',
    xytext=(8, 8)
)

plt.xlabel('Validation macro F1 (best threshold)')
plt.ylabel('test_sample macro F1 (selected strategy)')
plt.title('So sánh validation và test_sample của các ứng viên')
plt.legend()
plt.tight_layout()
plt.show()


### Hình cho report: confusion matrix của cấu hình được chọn trên `test_sample`

Hình này giúp tôi trình bày trực tiếp kiểu lỗi mà blend cuối cùng đang mắc phải trên tập có nhãn gần với pha đánh giá cuối.


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_ts, best_ts_preds, labels=[0, 1], normalize='true')

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, interpolation='nearest')
ax.set_title('Normalized confusion matrix on test_sample')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Human', 'AI'])
ax.set_yticklabels(['Human', 'AI'])
ax.set_xlabel('Predicted label')
ax.set_ylabel('True label')

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, f'{cm[i, j]:.3f}', ha='center', va='center')

fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
plt.tight_layout()
plt.show()


## 9. Refit các thành phần được chọn trên train + validation

Sau khi chốt blend, tôi chỉ refit những nhánh có trọng số khác 0 trên toàn bộ `train + val`. Mục đích của bước này là tận dụng thêm dữ liệu gán nhãn trước khi suy luận trên `test`, nhưng vẫn giữ nguyên kiến trúc và logic chọn blend đã quyết định ở trên.


In [10]:
def predict_probs_in_batches(clf, X_struct, X_text, batch_size=50000, r_text=None):
    probs = np.empty(X_struct.shape[0], dtype=np.float32)
    n = X_struct.shape[0]
    for start in range(0, n, batch_size):
        end = min(start + batch_size, n)
        Xb_text = X_text[start:end]
        if r_text is not None:
            Xb_text = Xb_text.multiply(r_text)
        Xb = hstack([X_struct[start:end], Xb_text], format='csr')
        probs[start:end] = clf.predict_proba(Xb)[:, 1].astype(np.float32)
        del Xb_text, Xb
        if start % (batch_size * 4) == 0:
            gc.collect()
    gc.collect()
    return probs

y_tv = np.concatenate([y_train, y_val]).astype(np.int32)
train_val_prior_ai = float(y_tv.mean())

use_base = best_base_w > 0
use_nb   = best_nb_w > 0
use_tree = best_tree_w > 0

base_test_probs = None
nb_test_probs = None
tree_test_probs = None

if use_base or use_nb:
    tv_feats_raw = pd.concat([train_feats_raw, val_feats_raw], axis=0).reset_index(drop=True)
    scaler_final = StandardScaler()
    tv_struct_scaled   = scaler_final.fit_transform(tv_feats_raw.values)
    test_struct_scaled_final = scaler_final.transform(test_feats_raw.values)

    X_tv_struct = csr_matrix(tv_struct_scaled.astype(np.float32))
    X_test_struct_final = csr_matrix(test_struct_scaled_final.astype(np.float32))
    X_tv_text = vstack([X_train_text, X_val_text], format='csr')

    if use_base:
        base_clf_final = SGDClassifier(
            loss='log_loss',
            alpha=1e-5,
            class_weight='balanced',
            max_iter=3000,
            random_state=42
        )
        print('Refitting final base model on train+val...')
        t0 = time.time()
        X_tv_base = hstack([X_tv_struct, X_tv_text], format='csr')
        base_clf_final.fit(X_tv_base, y_tv)
        print(f'Final base fit done in {(time.time()-t0)/60:.1f} min')
        del X_tv_base
        gc.collect()

        base_test_probs = predict_probs_in_batches(
            base_clf_final, X_test_struct_final, X_test_text, batch_size=50000
        )

    if use_nb:
        print('Refitting final NB-SVM model on train+val...')
        r_text_final = compute_log_count_ratio(X_tv_text, y_tv, alpha=1.0)
        nbsvm_clf_final = SGDClassifier(
            loss='log_loss',
            alpha=5e-6,
            class_weight='balanced',
            max_iter=3000,
            random_state=42
        )
        t0 = time.time()
        X_tv_nb = hstack([X_tv_struct, X_tv_text.multiply(r_text_final)], format='csr')
        nbsvm_clf_final.fit(X_tv_nb, y_tv)
        print(f'Final NB-SVM fit done in {(time.time()-t0)/60:.1f} min')
        del X_tv_nb
        gc.collect()

        nb_test_probs = predict_probs_in_batches(
            nbsvm_clf_final, X_test_struct_final, X_test_text, batch_size=50000, r_text=r_text_final
        )

    del X_tv_struct, X_test_struct_final, X_tv_text, tv_struct_scaled, test_struct_scaled_final
    gc.collect()

if use_tree:
    tv_feats_raw_tree = pd.concat([train_feats_raw, val_feats_raw], axis=0).reset_index(drop=True)
    final_rounds = int(max(tree_best_round, 80))
    tree_clf_final = make_tree_model(n_estimators=final_rounds, random_state=42)
    print(f'Refitting final tree model on train+val with n_estimators={final_rounds}...')
    t0 = time.time()
    tree_clf_final.fit(tv_feats_raw_tree, y_tv)
    print(f'Final tree fit done in {(time.time()-t0)/60:.1f} min')
    tree_test_probs = tree_clf_final.predict_proba(test_feats_raw)[:, 1].astype(np.float32)

test_probs = (
    best_base_w * base_test_probs +
    best_nb_w   * nb_test_probs +
    best_tree_w * tree_test_probs
).astype(np.float32)

if best_strategy == 'prior_corrected':
    test_probs_final = prior_correct(test_probs, train_val_prior_ai, test_prior_ai)
else:
    test_probs_final = test_probs

test_preds = (test_probs_final > best_thresh).astype(np.int32)

print(f'Test predictions (candidate={best_name}, strategy={best_strategy}, thresh={best_thresh:.3f}):')
print(f'  Human (0): {(test_preds == 0).sum():,} ({(test_preds == 0).mean()*100:.1f}%)')
print(f'  AI (1):    {(test_preds == 1).sum():,} ({(test_preds == 1).mean()*100:.1f}%)')
print(f'  Blend weights: base={best_base_w:.2f}, nbsvm={best_nb_w:.2f}, tree={best_tree_w:.2f}')


Refitting final base model on train+val...
Final base fit done in 0.8 min
Refitting final NB-SVM model on train+val...
Final NB-SVM fit done in 1.5 min
Refitting final tree model on train+val with n_estimators=500...
Final tree fit done in 0.7 min
Test predictions (candidate=blend_b0.4_n0.3_t0.3, strategy=prior_corrected, thresh=0.945):
  Human (0): 316,394 (63.3%)
  AI (1):    183,606 (36.7%)
  Blend weights: base=0.40, nbsvm=0.30, tree=0.30


## 10. Tạo file submission

Ở bước cuối, tôi áp dụng blend đã chọn lên tập `test`, chuyển xác suất thành nhãn nhị phân bằng threshold tương ứng, rồi ghi kết quả ra `submission.csv`.


In [11]:
test_ids = pd.read_parquet(f'{DATA_DIR}/test.parquet', columns=['ID'])

submission = pd.DataFrame({
    'ID': test_ids['ID'],
    'label': test_preds
})
submission.to_csv('submission.csv', index=False)

print(f'Submission saved: {submission.shape}')
print(submission['label'].value_counts())
display(submission.head())

# Free large objects before notebook conversion
del X_train_struct, X_val_struct, X_test_struct, X_ts_struct
del X_train_text, X_val_text, X_test_text, X_ts_text
gc.collect()


Submission saved: (500000, 2)
label
0    316394
1    183606
Name: count, dtype: int64


,ID,label
0,0,0
2,2,0
5,5,0
6,6,0
7,7,0


0

In [12]:
# Optional: uncomment to submit from Kaggle
# !kaggle competitions submit -c sem-eval-2026-task-13-subtask-a -f submission.csv -m "v6 linear + nbsvm + handcrafted tree ensemble"
